In [ ]:
import pyvista as pv

pv.set_jupyter_backend("static")

%load_ext autoreload
%autoreload 2

## Rosenthal solidification demo

Simulate an AM build using `RosenthalSolidificationModel`.

In [ ]:
from materialite import Material
from materialite.models import RosenthalSolidificationModel
from materialite.models.laser_path import LaserPath

Create a `Material` with an initially random field of grain IDs. The voxel size is 5 microns. `RosenthalSolidificationModel` expects a field with the name "grain" to be in the `Material` representing the initial grain ID of each voxel.

In [ ]:
material = Material(
    spacing=[5.0e-6, 5.0e-6, 5.0e-6], dimensions=[40, 90, 33]
).create_random_integer_field("grain", 1, 10000)

Create a two-layer `LaserPath`. The scan rotation is 67 degrees, and the layer thickness is 40 microns. The `z_height` of the first layer and the `end_offset` ensure that the melt pool does not extend beyond the bottom of the domain, and the melt pool moves out of the domain before the end of a scan.

In [ ]:
hatch_spacing = 110.0e-6
time_between_scans = 1.0e-5
path_inputs = {
    "x_bounds": [material.origin[0], material.sizes[0]],
    "y_bounds": [material.origin[1], material.sizes[1]],
    "hatch_spacing": hatch_spacing,
    "velocity": 0.96,
    "power": 285.0,
    "end_offset": 900.0e-6,
    "domain_offset": hatch_spacing / 2,
    "delay_between_scans": time_between_scans,
}
path1 = LaserPath.angled_raster_scan(angle=67.0, z_height=120.0e-6, **path_inputs)
path2 = LaserPath.angled_raster_scan(
    angle=134.0,
    z_height=160.0e-6,
    start_time=path1.total_time + time_between_scans,
    **path_inputs
)
laser_path = LaserPath.from_list([path1, path2])

Use a set of properties intended to represent Inconel 625. The time step duration ensures that the laser does not "jump" more than one voxel length. The output times tell the model to store the temperature field and grain IDs at the halfway point of each scan. The temperature field and grain IDs at the end of the simulation are also stored automatically.

In [ ]:
time_step_duration = 5.2e-6
output_times = (
    laser_path.end_times - laser_path.start_times
) / 2 + laser_path.start_times
thermophysical_properties = {
    "conductivity": 29.6,
    "diffusivity": 4.79e-6,
    "absorptivity": 0.46,
    "melt_temperature": 1609.0,
}
solidification_properties = {"prefactor": 1.2e-5, "exponent": 2.6}
initial_temperature = 300.0
process_model = RosenthalSolidificationModel(
    laser_path=laser_path,
    initial_temperature=300.0,
    thermophysical_properties=thermophysical_properties,
    solidification_properties=solidification_properties,
    time_step_duration=time_step_duration,
    output_times=output_times,
)
material = material.run(process_model)

Plot the final grain structure. There are unmelted areas in the bottom of the domain that the melt pool did not reach during the first layer. The grain structure at the top of the domain shows clear evidence of the scan directions.

In [ ]:
material.plot("grain")

Plot the temperature field during a scan in the first layer. The temperature is capped at the melting temperature.

In [ ]:
example_temperature = material.extract("temperature")[:, 2].components
material.create_fields({"example_temperature": example_temperature}).crop(
    z=[0, 120.0e-6]
).plot("example_temperature")

The `Material` has "temperature", "grain", and "grain_history" fields with simulation results.

In [ ]:
material.get_fields()

The temperature field is stored as a Materialite `Scalar` object with a points dimension and a time dimension. This contains the pointwise temperature history at the `output_times` provided to the model, as well as the end of the simulation if it was not included in the list of `output_times`.

In [ ]:
material.extract("temperature")

The "grain" field contains the grain IDs at the end of the simulation. The "grain_history" field contains the grain IDs at each of the `output_times` (and the end of the simulation) as a Materialite `Scalar` object, just like the "temperature" field. The reason for the two separate grain fields is that it is often useful to have a field containing only the current grain IDs for subsequent operations (e.g., adding orientations, visualizing an IPF map, and setting up a mechanical simulation).

In [ ]:
material.extract("grain_history")

In [ ]:
material.extract("grain")